In [4]:
import pandas as pd

# 'your_file_name.csv' ki jagah apni uploaded file ka exact naam likhein
df = pd.read_csv('/content/tmdb_5000_movies.csv')

print("Google Colab par Setup 100% Ready Hai!")
print("\n--- Data ke columns ye hain: ---")
print(df.columns)


Google Colab par Setup 100% Ready Hai!

--- Data ke columns ye hain: ---
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')


In [5]:
import pandas as pd

# Load the credits dataset
credits_df = pd.read_csv('/content/tmdb_5000_credits.csv')

# Merge both datasets on the 'title' column
movies = df.merge(credits_df, on='title')

# Print the shape to verify rows and columns
print("Merged successfully! Dataset shape:", movies.shape)


Merged successfully! Dataset shape: (4809, 23)


In [6]:
# Keep only the columns needed for content-based recommendation
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

# Drop rows that contain missing values (NaN)
movies.dropna(inplace=True)

# Verify the updated columns
print("Columns selected and missing values dropped.")


Columns selected and missing values dropped.


In [7]:
import ast

# Function to extract all names from a JSON-like string list
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

# Apply conversion to genres and keywords
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

# Function to extract the first 3 actor names from the cast
def convert3(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
        counter += 1
    return L

movies['cast'] = movies['cast'].apply(convert3)

# Function to extract only the Director name from the crew
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

# Convert the overview text into a list of individual words
movies['overview'] = movies['overview'].apply(lambda x: x.split())

print("Data parsing and formatting completed.")


Data parsing and formatting completed.


In [8]:
# Remove spaces between words to prevent split-word mismatch
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ","") for i in x])

# Concatenate all list columns into a single 'tags' column
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Create a new simplified dataframe with only essential columns
new_df = movies[['movie_id', 'title', 'tags']]

# Convert the list of tags back into a single lowercased string paragraph
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

# Preview the final prepared dataframe
new_df.head()


/tmp/ipykernel_3993/1853366184.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
/tmp/ipykernel_3993/1853366184.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [9]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer to keep top 5000 frequent words and remove stop words (like 'and', 'the', 'is')
cv = CountVectorizer(max_features=5000, stop_words='english')

# Convert the text tags into a numerical matrix (vectors)
vectors = cv.fit_transform(new_df['tags']).toarray()

# Check the shape of our vector matrix
print("Vectorization complete! Matrix shape:", vectors.shape)


Vectorization complete! Matrix shape: (4806, 5000)


In [10]:
import nltk
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

# Function to stem each word in the tags
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

# Apply stemming to the tags column
new_df['tags'] = new_df['tags'].apply(stem)

# Re-run vectorization on the updated stemmed tags
vectors = cv.fit_transform(new_df['tags']).toarray()
print("Stemming and Re-vectorization successfully applied.")


/tmp/ipykernel_3993/1630135529.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


Stemming and Re-vectorization successfully applied.


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the cosine similarity between all movie vectors
similarity = cosine_similarity(vectors)

# Check the shape (It will be 4806 x 4806, matching every movie with every other movie)
print("Similarity matrix calculated! Shape:", similarity.shape)


Similarity matrix calculated! Shape: (4806, 4806)


In [12]:
def recommend(movie):
    # Fetch the index of the movie provided by the user
    try:
        movie_index = new_df[new_df['title'] == movie].index[0]

        # Get similarity scores for this specific movie
        distances = similarity[movie_index]

        # Sort the similarity scores in descending order and fetch top 5 items (excluding the movie itself)
        movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

        print(f"Recommendations for '{movie}':\n")
        # Print the titles of the recommended movies
        for i in movies_list:
            print(new_df.iloc[i[0]].title)

    except IndexError:
        print(f"Movie '{movie}' not found in the dataset. Please check the spelling.")

# TEST THE MODEL: Try passing a movie name present in your dataset
recommend('Avatar')


Recommendations for 'Avatar':

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [23]:
import pickle

# Export the movie list dataframe as a dictionary file
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))

# Export the similarity matrix file
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("Files exported successfully!")


Files exported successfully!


In [14]:
# Install required deployment libraries cleanly on Google Colab server
!pip install -q streamlit
!npm install -q localtunnel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 40.7 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 3s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [15]:
%%writefile app.py
import streamlit as st
import pandas as pd

# Function to recommend top 5 similar movies
def recommend(movie):
    movie_index = movies[movies['title'] == movie].index
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x)[1:6]

    recommended_movies = []
    for i in movies_list:
        recommended_movies.append(movies.iloc[i].title)
    return recommended_movies

# Web Page UI Styling Configuration
st.set_page_config(page_title="Movie Recommender", page_icon="🎬")
st.title('Movie Recommendation System 🎬')
st.write("Select a movie you like, and we will recommend 5 similar movies for you!")

# Securely bind global notebook dataframe and model variables
import __main__
movies = __main__.new_df
similarity = __main__.similarity

# Dropdown list element mapping
movie_list = movies['title'].values
selected_movie_name = st.selectbox('Type or select a movie from the dropdown:', movie_list)

# Render results upon clicking the button interface
if st.button('Recommend'):
    recommendations = recommend(selected_movie_name)
    st.subheader('Top 5 Recommended Movies:')
    for index, movie in enumerate(recommendations, 1):
        st.write(f"**{index}.** {movie}")


Writing app.py


In [16]:
import urllib.request

# Programmatically query the server environment IP address
tunnel_password = urllib.request.urlopen('https://icanhazip.com').read().decode('utf8').strip()

print("Your Tunnel Password (IP Address) is:")
print(tunnel_password)


Your Tunnel Password (IP Address) is:
104.199.189.81


In [17]:
# Launch Streamlit server background routine linked with localtunnel routing
!streamlit run app.py & npx localtunnel --port 8501


⠙⠹your url is: https://tidy-mugs-hide.loca.lt


2026-06-20 11:59:28.252 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://104.199.189.81:8501

  Stopping...
^C


In [18]:
import pickle

# Save the movie data as a clean CSV file
new_df.to_csv('movies_clean.csv', index=False)

# Save the similarity matrix safely as a binary pickle file
with open('similarity_matrix.pkl', 'wb') as f:
    pickle.dump(similarity, f)

print("Both data files generated and saved successfully!")


Both data files generated and saved successfully!


In [19]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pickle

# Function to calculate and recommend top 5 similar movies
def recommend(movie):
    movie_index = movies[movies['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    recommended_movies = []
    for i in movies_list:
        recommended_movies.append(movies.iloc[i[0]].title)
    return recommended_movies

# Webpage Layout Configuration
st.set_page_config(page_title="Movie Recommender", page_icon="🎬")
st.title('Movie Recommendation System 🎬')
st.write("Select a movie you like, and we will recommend 5 similar movies for you!")

# Load the local files cleanly to prevent main data mismatch errors
movies = pd.read_csv('movies_clean.csv')
with open('similarity_matrix.pkl', 'rb') as f:
    similarity = pickle.load(f)

# Map movie titles to the dropdown widget
movie_list = movies['title'].values
selected_movie_name = st.selectbox('Type or select a movie from the dropdown:', movie_list)

# Handle the click action on the recommend button interface
if st.button('Recommend'):
    recommendations = recommend(selected_movie_name)
    st.subheader('Top 5 Recommended Movies:')
    for index, movie in enumerate(recommendations, 1):
        st.write(f"**{index}.** {movie}")


Overwriting app.py


In [20]:
import urllib.request

# Fetch the network IP address that acts as the access key
tunnel_password = urllib.request.urlopen('https://icanhazip.com').read().decode('utf8').strip()

print("Your Tunnel Password (IP Address) is:")
print(tunnel_password)


Your Tunnel Password (IP Address) is:
104.199.189.81


In [ ]:
# Launch background streamlit server connected with localtunnel routing port
!streamlit run app.py & npx localtunnel --port 8501


⠙⠹

your url is: https://fluffy-falcons-look.loca.lt
2026-06-20 12:03:34.221 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://104.199.189.81:8501

